In [ ]:
%load_ext autoreload
import os
import numpy as np
import pandas as pd

import torch

import libraries.predict_number as pn

In [13]:
from libraries.data_loader import load_datas

path = r"datas/brut/mnist.npz"
keys = ['x_train', 'y_train', 'x_test', 'y_test']

_, Y_TRAIN, _, Y_VAL = load_datas(path, keys)

In [36]:
latent_train_path = r"datas\brut\latent_train.csv"
latent_test_path = r"datas\brut\latent_test.csv"
latent_bn_path = r"datas\bn_sample\rbn.csv"


z_act_mix_train = pd.read_csv(latent_train_path)
z_act_mix = pd.read_csv(latent_test_path)
tilde_z_act_mix = pd.read_csv(latent_bn_path, index_col=0).iloc[:, :23]
tilde_z_act_mix.columns = [int(col.split("_")[-1]) for col in tilde_z_act_mix.columns]

Y_BN = pd.read_csv(latent_bn_path)["number"]

In [37]:
tilde_z_act_mix.head(5)

,24,36,3,25,21,33,26,43,38,40,...,22,4,10,41,20,31,27,19,14,39
1,0.566217,-1.170342,-2.250670,-0.179925,0.132216,-0.735915,-0.261937,1.540262,0.174024,-0.200603,...,-1.347865,-0.183022,0.908865,-0.245996,-0.603323,-0.084503,-0.202100,-0.405540,0.379030,-0.183458
2,0.412737,0.762277,-0.134814,2.596862,-1.498112,-0.519310,0.935995,-0.370478,-0.626781,0.947257,...,-0.511473,-0.687745,-0.272937,-0.032261,-0.445845,-0.434047,-0.351202,-0.209977,-0.449020,-0.377353
3,1.177984,0.850908,0.715987,-0.036559,-2.455330,0.006973,-0.048226,-0.587261,-1.200089,-0.682814,...,0.172799,-1.518209,0.293541,-0.484070,-0.724939,0.990778,1.294003,-0.071927,-0.776211,-0.091745
4,-2.974064,2.265716,0.822757,1.662648,-0.763649,-0.760459,-0.022402,-0.355798,0.332719,-0.327818,...,-0.259913,-0.155845,0.458226,-0.043329,-0.601366,0.241106,-0.197574,0.194641,0.213842,-0.503613
5,-0.320750,0.447855,-1.446682,-1.008489,-0.806471,-0.663810,1.301289,-0.308024,1.251331,1.098929,...,-0.931200,-0.614105,0.318659,-0.565891,1.019865,-0.032512,0.267601,0.474702,-0.656998,0.195872


In [6]:
z_act_mix.head(5)

,24,36,3,25,21,33,26,43,38,40,...,22,4,10,41,20,31,27,19,14,39
0,-0.167848,0.877249,-2.419570,-0.821540,0.789623,0.537393,0.979894,0.630535,0.551719,1.271499,...,0.244937,-0.033302,0.021839,-0.334690,0.058168,0.147661,-0.285290,-0.241294,0.127128,-0.116712
1,-0.513359,-0.561446,1.305558,0.293582,-2.970291,-0.154321,-1.064934,-1.315598,-0.329532,-1.087535,...,-0.677390,-1.133839,-1.025260,0.566719,-0.811193,-0.740626,-0.581199,1.065670,0.218937,0.029720
2,-2.200929,-1.119576,0.200688,0.095202,0.164410,0.877677,0.397622,0.133279,-0.303290,-0.124841,...,-0.274826,0.031830,0.010194,0.020773,-0.039979,0.053413,-0.141140,0.106036,0.122870,-0.167953
3,0.953803,0.197884,-0.237537,1.019155,0.019350,-1.470674,1.474770,-0.500433,0.176988,-0.928749,...,0.311281,-0.187279,-0.115489,0.540916,0.425460,0.183578,0.429480,0.284673,-0.151406,0.155518
4,0.460676,1.400474,0.658669,0.622022,0.281464,1.199245,-0.799182,1.182759,-0.856209,0.798141,...,0.753601,0.440321,-0.275433,0.029788,-0.386370,-0.282071,-0.483767,-0.089588,-0.809915,-0.322145


In [44]:
V_act_mix = z_act_mix.columns
d = len(V_act_mix)
learning_rate = 1e-3
batchsize = 200
epochs = 10
cur_epoch = 1

net = pn.Predicter(d)
torch.manual_seed(42)
optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate, weight_decay=1e-4)

In [45]:

batch_train = range(0, z_act_mix_train.shape[0] +1, batchsize) 
batch_val = range(0, z_act_mix.shape[0] +1, batchsize)
batch_samp = range(0, tilde_z_act_mix.shape[0] +1, batchsize) 
ratio = 100/batchsize


AVG_loss = []
AVG_loss_val = []
AVG_loss_samp = []
str_losses = ["loss", "val_loss", "samp_loss", "loss_pred", "val_loss_pred", "samp_loss_pred"]
for epoch in range(epochs) :
    avg_loss = []
    avg_val_loss = []
    avg_samp_loss = []
    avg_loss_pred = []
    avg_val_loss_pred = []
    avg_samp_loss_pred = []
    for b, batch in enumerate(batch_train[:-1]) :
        Z = z_act_mix_train[batch:batch_train[b+1]]
        Y = Y_TRAIN[batch:batch_train[b+1]]

        Z = torch.from_numpy(np.array(Z, dtype = np.float32))
        loss, loss_pred = net(Z, Y)
        
        # Rétropropagation du gradient
        optimizer.zero_grad()
        loss.backward()
        # Descente de gradient (une itération)
        optimizer.step()
        
        avg_loss.append(loss.item()*ratio)
        avg_loss_pred.append(loss_pred.item()*ratio)

    for b, batch in enumerate(batch_val[:-1]) :
        Z = z_act_mix[batch:batch_train[b+1]]
        Y = Y_VAL[batch:batch_train[b+1]]

        Z = torch.from_numpy(np.array(Z, dtype = np.float32))
        val_loss, val_loss_pred = net(Z, Y)
        avg_val_loss.append(val_loss.item() *ratio)
        avg_val_loss_pred.append(val_loss_pred.item() *ratio)

    for b, batch in enumerate(batch_samp[:-1]) :
        Z = tilde_z_act_mix[batch:batch_samp[b+1]]
        Y = Y_BN[batch:batch_samp[b+1]]

        Z = torch.from_numpy(np.array(Z, dtype = np.float32))
        val_loss, val_loss_pred = net(Z, Y)
        avg_samp_loss.append(val_loss.item() *ratio)
        avg_samp_loss_pred.append(val_loss_pred.item() *ratio)

    
    
    losses = [ avg_loss, avg_val_loss, avg_samp_loss, avg_loss_pred, avg_val_loss_pred, avg_samp_loss_pred]
    print_losses = " - ".join([f"{strloss}: {np.round(np.mean(loss), 3)}" for strloss, loss in zip(str_losses, losses)])
    print(f"{epoch +1:03d} {print_losses}")
    # clear_output()
    AVG_loss.append([avg_loss, avg_loss_pred])
    AVG_loss_val.append([avg_val_loss, avg_val_loss_pred])
    AVG_loss_samp.append([avg_samp_loss, avg_samp_loss_pred])

AVG_loss = np.concatenate(AVG_loss, axis = 1).T
AVG_loss_val = np.concatenate(AVG_loss_val, axis = 1).T
AVG_loss_samp = np.concatenate(AVG_loss_samp, axis = 1).T
df = pd.DataFrame(AVG_loss, columns = [str_losses[0], str_losses[3]])
df_val = pd.DataFrame(AVG_loss_val, columns = [str_losses[1], str_losses[4]])
df_samp = pd.DataFrame(AVG_loss_samp, columns = [str_losses[2], str_losses[5]])

001 loss: 77.798 - val_loss: 22.852 - samp_loss: 110.32 - loss_pred: 90.687 - val_loss_pred: 97.33 - samp_loss_pred: 81.15
002 loss: 17.589 - val_loss: 14.443 - samp_loss: 114.242 - loss_pred: 97.79 - val_loss_pred: 98.15 - samp_loss_pred: 81.12
003 loss: 11.932 - val_loss: 11.988 - samp_loss: 119.451 - loss_pred: 98.53 - val_loss_pred: 98.34 - samp_loss_pred: 81.41
004 loss: 9.51 - val_loss: 10.92 - samp_loss: 125.778 - loss_pred: 98.817 - val_loss_pred: 98.48 - samp_loss_pred: 81.31
005 loss: 8.013 - val_loss: 10.258 - samp_loss: 131.768 - loss_pred: 99.03 - val_loss_pred: 98.57 - samp_loss_pred: 81.0
006 loss: 6.917 - val_loss: 9.882 - samp_loss: 137.347 - loss_pred: 99.18 - val_loss_pred: 98.6 - samp_loss_pred: 81.01
007 loss: 6.058 - val_loss: 9.724 - samp_loss: 143.264 - loss_pred: 99.29 - val_loss_pred: 98.63 - samp_loss_pred: 80.83
008 loss: 5.351 - val_loss: 9.659 - samp_loss: 149.393 - loss_pred: 99.365 - val_loss_pred: 98.67 - samp_loss_pred: 80.69
009 loss: 4.754 - val_loss

In [46]:
predicter_path = r"results\Predicter\Predicter_from_z.pt"
torch.save(net, predicter_path)